# OSS4008A — Hospital Infrastructure Detection

**Chunk 1: Earth Engine connectivity and a Halifax imagery preview.**

This notebook is a runnable starting point, **not** a trained hospital detector. Use Google Colab for official course work. Authenticate using your own Google account. No tokens or secrets are stored here.

In [ ]:
import os
import sys
import importlib.metadata as metadata

IN_COLAB = "google.colab" in sys.modules
print("Running in:", "Google Colab" if IN_COLAB else "Local Python / VS Code")
if IN_COLAB:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "earthengine-api>=1.5,<2", "geemap>=0.35,<1"])
else:
    print("For local setup, run in a terminal: python -m pip install -r code/requirements.txt")

In [ ]:
import ee
import geemap

print("earthengine-api:", metadata.version("earthengine-api"))
print("geemap:", metadata.version("geemap"))

## Authenticate Google Earth Engine (local VS Code / Google Colab)

This step requires **Google account authorization**. The application cannot grant itself access. For Windows + VS Code the `localhost` authorization mode launches a browser and listens for the redirect. Colab uses its normal authentication flow.

The Google Cloud project ID below came from the previous OSS 4008 lab notebooks. Successful initialization still depends on your Google account having permission to use that project. No credentials are stored in the notebook.

**Run this cell after the imports.** If a Google sign-in or consent page opens, finish it in your browser. After that the notebook verifies access with a small server-side request.


In [ ]:
import ee
import os
import sys

EE_PROJECT_ID = os.environ.get("EE_PROJECT_ID", "infra-bedrock-411215").strip()
if not EE_PROJECT_ID:
    raise ValueError("EE_PROJECT_ID must be set to an Earth Engine-enabled Cloud project ID.")

IN_COLAB = "google.colab" in sys.modules

print("Authenticating Earth Engine for:", "Colab" if IN_COLAB else "Windows / VS Code")
print("Project:", EE_PROJECT_ID)

# Local Jupyter may hang if it uses the Colab-oriented authentication flow.
# Select the explicit localhost flow on your own Windows computer.
if IN_COLAB:
    ee.Authenticate()
else:
    ee.Authenticate(auth_mode="localhost")

print("Authentication step finished. Initializing...")
ee.Initialize(project=EE_PROJECT_ID)
print("Earth Engine server test:", ee.Number(42).getInfo())
print("Earth Engine initialized successfully for:", EE_PROJECT_ID)


## Halifax basemap (Sentinel-2)

This is an **imagery connectivity demonstration**. Sentinel-2 is generally 10 m resolution in its visible bands, which is not sufficient on its own for reliable hospital-building identification. Selecting a suitably fine-resolution image source is a later milestone.

In [ ]:
HALIFAX_LAT, HALIFAX_LON = 44.6488, -63.5752
point = ee.Geometry.Point([HALIFAX_LON, HALIFAX_LAT])
collection = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
              .filterBounds(point)
              .filterDate("2025-06-01", "2025-09-30")
              .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20)))
count = collection.size().getInfo()
print("Candidate Sentinel-2 scenes:", count)
if count == 0:
    raise RuntimeError("No images match the filters. Expand dates or cloud threshold.")
image = collection.median()
m = geemap.Map(center=[HALIFAX_LAT, HALIFAX_LON], zoom=11)
m.addLayer(image, {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000}, "Sentinel-2 true color")
m.addLayerControl()
display(m)

## Initial verification

**Success:** initialization prints the project ID, a positive candidate scene count, and an interactive Halifax map. **Not yet complete:** hospital labels, detection model, confidence evaluation, and chatbot. Save a screenshot in `results/` after executing this notebook in Colab.